# 🎯 Face Recognition Module

Verify voter identity by comparing live face with registered face embeddings.

**Pipeline:**
1. Face Detection → Detect and crop face
2. Face Embedding → Extract 512-dim feature vector
3. Face Verification → Compare embeddings using cosine similarity

## Step 1: Install Dependencies

In [1]:
# Install InsightFace for face recognition
!pip install insightface onnxruntime

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 4.6 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 5.7 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 5.0 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 5.4 MB/s  0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp313-cp313-macosx_10_13_universal2.whl size=927576 sha256=f1f99a6369c09d6a03523212047d76012ebfe8c81837b5a8f030619ef94e2d30
  Stored in directory: /Users/hemanthtavva/Library/Caches/pip/wheels/e8/a1/6a/a94e7156c3c0815101a10be114c44fc0b2b3e40fc10d63b939
Successfully built insightface
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [insightface] [albumentations]eadless]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip instal

## Step 2: Initialize Face Recognition Model

In [1]:
import os
import cv2
import numpy as np
from pathlib import Path

import insightface
from insightface.app import FaceAnalysis

# Create models directory
os.makedirs('../models/face_recognition', exist_ok=True)

# Initialize InsightFace with ArcFace model
# This will download the model automatically on first run
print("Loading InsightFace model (this may take a moment on first run)...")

app = FaceAnalysis(
    name='buffalo_l',  # Best accuracy model
    providers=['CPUExecutionProvider']  # Use CPU (works on all systems)
)

# Prepare the model (ctx_id=-1 for CPU, 0 for GPU)
app.prepare(ctx_id=-1, det_size=(640, 640))

print("\n✅ InsightFace model loaded!")
print("   - Detection: RetinaFace")
print("   - Recognition: ArcFace (512-dim embeddings)")

Loading InsightFace model (this may take a moment on first run)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/hemanthtavva/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/hemanthtavva/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/hemanthtavva/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/hemanthtavva/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/hema

## Step 3: Define Face Embedding Functions

In [2]:
def get_face_embedding(image, app):
    """
    Extract face embedding from an image.
    
    Args:
        image: BGR image (OpenCV format)
        app: InsightFace FaceAnalysis instance
    
    Returns:
        embedding: 512-dim numpy array, or None if no face detected
        face_info: dict with bbox, landmarks, etc.
    """
    # Detect faces
    faces = app.get(image)
    
    if len(faces) == 0:
        return None, None
    
    # Get the largest face (in case multiple faces detected)
    face = max(faces, key=lambda x: (x.bbox[2] - x.bbox[0]) * (x.bbox[3] - x.bbox[1]))
    
    # Extract embedding (512-dim vector)
    embedding = face.embedding
    
    # Normalize embedding
    embedding = embedding / np.linalg.norm(embedding)
    
    face_info = {
        'bbox': face.bbox.astype(int),
        'score': face.det_score,
    }
    
    return embedding, face_info


def compare_faces(embedding1, embedding2, threshold=0.5):
    """
    Compare two face embeddings using cosine similarity.
    
    Args:
        embedding1: First face embedding (512-dim)
        embedding2: Second face embedding (512-dim)
        threshold: Similarity threshold (default 0.5)
    
    Returns:
        similarity: float between -1 and 1 (higher = more similar)
        is_match: bool based on threshold
    """
    # Cosine similarity
    similarity = np.dot(embedding1, embedding2)
    
    is_match = similarity > threshold
    
    return similarity, is_match


print("✅ Face embedding functions defined")
print("   - get_face_embedding(): Extract 512-dim embedding from image")
print("   - compare_faces(): Compare two embeddings (cosine similarity)")

✅ Face embedding functions defined
   - get_face_embedding(): Extract 512-dim embedding from image
   - compare_faces(): Compare two embeddings (cosine similarity)


## Step 4: Face Registration (Capture & Store Embedding)

For testing, we'll capture your face from webcam and save the embedding locally.

In [6]:
import json
import time

# Create directory for registered faces
REGISTERED_FACES_DIR = '../data/registered_faces'
os.makedirs(REGISTERED_FACES_DIR, exist_ok=True)

def register_face(voter_id, app, num_captures=3):
    """
    Register a voter's face by capturing from webcam.
    
    Args:
        voter_id: Unique identifier (e.g., Aadhaar number or test ID)
        app: InsightFace FaceAnalysis instance
        num_captures: Number of face samples to capture and average
    
    Returns:
        success: bool
        message: str
    """
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        return False, "Could not open webcam"
    
    print("=" * 60)
    print(f"FACE REGISTRATION - Voter ID: {voter_id}")
    print("=" * 60)
    print(f"Will capture {num_captures} samples. Look at the camera.")
    print("Press SPACE to capture, 'q' to quit")
    print("=" * 60)
    
    embeddings = []
    captured = 0
    
    while captured < num_captures:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Detect face and draw box
        faces = app.get(frame)
        display_frame = frame.copy()
        
        if len(faces) > 0:
            face = max(faces, key=lambda x: (x.bbox[2] - x.bbox[0]) * (x.bbox[3] - x.bbox[1]))
            bbox = face.bbox.astype(int)
            cv2.rectangle(display_frame, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)
            cv2.putText(display_frame, f"Face detected - Press SPACE ({captured}/{num_captures})", 
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        else:
            cv2.putText(display_frame, "No face detected", (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        cv2.imshow('Face Registration', display_frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord(' ') and len(faces) > 0:  # SPACE to capture
            embedding, _ = get_face_embedding(frame, app)
            if embedding is not None:
                embeddings.append(embedding)
                captured += 1
                print(f"✅ Captured {captured}/{num_captures}")
                time.sleep(0.5)  # Brief pause between captures
        elif key == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    
    if len(embeddings) < num_captures:
        return False, f"Only captured {len(embeddings)}/{num_captures} samples"
    
    # Average the embeddings for better accuracy
    avg_embedding = np.mean(embeddings, axis=0)
    avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)  # Normalize
    
    # Save embedding
    save_path = os.path.join(REGISTERED_FACES_DIR, f"{voter_id}.npy")
    np.save(save_path, avg_embedding)
    
    print("\n" + "=" * 60)
    print(f"✅ REGISTRATION COMPLETE!")
    print(f"   Voter ID: {voter_id}")
    print(f"   Embedding saved to: {save_path}")
    print("=" * 60)
    
    return True, save_path


print("✅ Registration function defined")
print("   Run: register_face('test_user', app)")

✅ Registration function defined
   Run: register_face('test_user', app)


In [7]:
# Register your face - Press SPACE 3 times to capture
register_face('test_user', app)

2025-12-21 16:27:57.788 Python[21177:277335] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


FACE REGISTRATION - Voter ID: test_user
Will capture 3 samples. Look at the camera.
Press SPACE to capture, 'q' to quit
✅ Captured 1/3
✅ Captured 2/3
✅ Captured 3/3

✅ REGISTRATION COMPLETE!
   Voter ID: test_user
   Embedding saved to: ../data/registered_faces/test_user.npy


(True, '../data/registered_faces/test_user.npy')

## Step 5: Face Verification (Live Webcam Test)

Compare live face with registered embedding to verify identity.

In [8]:
def verify_face(voter_id, app, threshold=0.5, duration=30):
    """
    Verify a voter's face against their registered embedding.
    
    Args:
        voter_id: Voter ID to verify against
        app: InsightFace FaceAnalysis instance
        threshold: Similarity threshold (0.5 = 50%)
        duration: Test duration in seconds
    """
    # Load registered embedding
    embedding_path = os.path.join(REGISTERED_FACES_DIR, f"{voter_id}.npy")
    if not os.path.exists(embedding_path):
        print(f"❌ No registered face found for voter: {voter_id}")
        return False
    
    registered_embedding = np.load(embedding_path)
    
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Could not open webcam")
        return False
    
    print("=" * 60)
    print(f"FACE VERIFICATION - Voter ID: {voter_id}")
    print("=" * 60)
    print(f"Threshold: {threshold*100:.0f}%")
    print(f"Running for {duration} seconds... Press 'q' to quit")
    print("=" * 60)
    
    start_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Get live face embedding
        live_embedding, face_info = get_face_embedding(frame, app)
        
        if live_embedding is not None:
            # Compare with registered embedding
            similarity, is_match = compare_faces(registered_embedding, live_embedding, threshold)
            
            # Draw results
            bbox = face_info['bbox']
            color = (0, 255, 0) if is_match else (0, 0, 255)
            label = "MATCH" if is_match else "NO MATCH"
            
            cv2.rectangle(frame, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
            cv2.putText(frame, f"{label}: {similarity*100:.1f}%", (bbox[0], bbox[1]-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
            cv2.putText(frame, f"Voter: {voter_id}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, f"Similarity: {similarity*100:.1f}%", (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, f"Threshold: {threshold*100:.0f}%", (10, 90),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        else:
            cv2.putText(frame, "No face detected", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 165, 255), 2)
        
        cv2.imshow('Face Verification', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n⏹️ Stopped by user")
            break
        if time.time() - start_time > duration:
            print(f"\n⏱️ Test completed ({duration}s)")
            break
    
    cap.release()
    cv2.destroyAllWindows()
    return True


print("✅ Verification function defined")
print("   Run: verify_face('test_user', app)")

✅ Verification function defined
   Run: verify_face('test_user', app)


In [11]:
# Verify your face against registered embedding
verify_face('test_user', app)

FACE VERIFICATION - Voter ID: test_user
Threshold: 50%
Running for 30 seconds... Press 'q' to quit

⏹️ Stopped by user


True